#### Scrapy 실습
- https://www.ikea.com/kr/ko/cat/lower-price/
    - 각 제품의 상세 정보
- https://www.melon.com/chart/index.htm
    - 각 노래의 상세 정보
- https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1
    - 각 도서의 상세 정보
- https://news.naver.com/section/105
    - 각 뉴스의 상세 정보
- 이외 링크를 기준으로 각 상세 정보를 획득할 수 있는 사이트 크롤링


In [1]:
%%capture
# scrapy 설치
!pip install scrapy

In [2]:
# scrapy, CrawlerProcess 모듈 불러오기
import scrapy
from scrapy.crawler import CrawlerProcess

In [3]:
class IkeaItem(scrapy.Item):
  name = scrapy.Field()
  use = scrapy.Field()
  price = scrapy.Field()
  size = scrapy.Field()
  # color = scrapy.Field()

In [4]:
class IkeaSpider(scrapy.Spider):
  name = 'ikea'

  def start_requests(self):
    url = "https://www.ikea.com/kr/ko/cat/lower-price/#product-list-skip"
    yield scrapy.Request(url, self.parse_start)

  def parse_start(self, response):
    product_links = set(response.css('div.plp-product-list__products a::attr(href)').getall())
    for i in product_links:
      yield scrapy.Request(i, self.parse_items)

  def parse_items(self, response):
    item = IkeaItem()
    item['name'] = response.xpath('//*[@id="pip-buy-module-content"]/div[1]/div[1]/div[1]/div[2]/span/h1/div/div[1]/span[1]/text()').get()
    item['price'] = response.xpath('//*[@id="pip-buy-module-content"]/div[1]/div[1]/div[2]/div/span/span[1]/span/span[2]/text()').get()
    item['size'] = response.xpath('//*[@id="pip-buy-module-content"]/div[1]/div[1]/div[1]/div[2]/span/h1/div/div[1]/span[2]/button/text()').get()
    item['use'] = response.xpath('//*[@id="pip-buy-module-content"]/div[1]/div[1]/div[1]/div[2]/span/h1/div/div[1]/span[2]/span/text()').get()
    # item['color'] = ''
    yield item

In [5]:
# class GenrePipeline:
#     def process_item(self, item, spider):
#          # 용도 및 모양 나누기
#          use_split = item['use'].split(',')
#          for i in range(0,len(use_split),2) :
#             item['use'] = use_split[i]
#             item['color'] = use_split[i+1]
#             if i + 1 < len(use_split):  # 인덱스 초과 방지
#                 item['color']= use_split[i + 1]
#             else:
#                 item['color']= 'Unknown'  # 짝이 없을 경우

In [6]:
process = CrawlerProcess(settings={
      "FEEDS" : {"ikea_product.csv": {"format": "csv", "encoding":"utf-8"}},
      "DOWNLOAD_DELAY" : 1.0,
      # "ITEM_PIPELINES": {
      #     '__main__.GenrePipeline' : 1
      # }
})

process.crawl(IkeaSpider)
process.start()

INFO:scrapy.utils.log:Scrapy 2.12.0 started (bot: scrapybot)
2025-02-03 07:14:35 [scrapy.utils.log] INFO: Scrapy 2.12.0 started (bot: scrapybot)
INFO:scrapy.utils.log:Versions: lxml 5.3.0.0, libxml2 2.12.9, cssselect 1.2.0, parsel 1.10.0, w3lib 2.3.1, Twisted 24.11.0, Python 3.11.11 (main, Dec  4 2024, 08:55:07) [GCC 11.4.0], pyOpenSSL 24.2.1 (OpenSSL 3.3.2 3 Sep 2024), cryptography 43.0.3, Platform Linux-6.1.85+-x86_64-with-glibc2.35
2025-02-03 07:14:35 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.12.9, cssselect 1.2.0, parsel 1.10.0, w3lib 2.3.1, Twisted 24.11.0, Python 3.11.11 (main, Dec  4 2024, 08:55:07) [GCC 11.4.0], pyOpenSSL 24.2.1 (OpenSSL 3.3.2 3 Sep 2024), cryptography 43.0.3, Platform Linux-6.1.85+-x86_64-with-glibc2.35
INFO:scrapy.addons:Enabled addons:
[]
2025-02-03 07:14:35 [scrapy.addons] INFO: Enabled addons:
[]
DEBUG:scrapy.utils.log:Using reactor: twisted.internet.epollreactor.EPollReactor
2025-02-03 07:14:35 [scrapy.utils.log] DEBUG: Using reactor: twi

In [ ]:
# 알라딘 가격 참고
#item['정가'] = response.xpath('//*[@id="Ere_prod_allwrap"]/div[4]/div[4]/div/div[position() >= 1 and position() <= 4]/ul/li[1]/div[2]/text()').get()